# Welcome to the AI Agentics Course

Welcome! In this course, we will discover, step by step, the basic building blocks of **agentic AI**: how to talk to an LLM, how to give it tools, how to chain several LLMs together, and eventually how to let an agent act on its own to complete a task. Each part of the course builds on the previous one, and we will use a running example — a philosophy professor who writes an assignment, has it answered by several AI "students", and then grades the results — to make the concepts concrete.

## What is an LLM?

A **Large Language Model (LLM)** is a program trained on huge amounts of text that has learned to predict, one piece of word (a *token*) at a time, what is likely to come next. Given a prompt, it generates a response by repeatedly predicting the next token until it produces a full answer. It has no memory between separate requests, no access to the outside world, and no built-in ability to act — it only reads text and produces text. Everything we call "agentic AI" is built on top of this simple mechanism: we structure prompts, chain calls, and connect LLMs to tools and to each other so that this text-in/text-out engine can carry out more useful, multi-step tasks.

Well-known examples include GPT (OpenAI), Gemini (Google), Claude (Anthropic), and open-source models such as Mistral or Llama that you can run yourself with Ollama. In this course we will access several of them, always through the same kind of API.

## Today: Part 1 — Setting up the tools

Before writing any agent, we need a working environment. Today's session is dedicated to installing and configuring the tools we will rely on throughout the course:

* **uv**, to manage our Python environment and dependencies,
* **Ollama**, to run an LLM locally on your own machine,
* API keys for the **University of Rennes** and **Google Gemini** servers,

and, most importantly, understanding **how we will query these different LLM servers**: whatever the provider (Rennes, Gemini, Ollama, or later OpenAI/Anthropic), we will always speak to it through the same OpenAI-compatible API, only changing the server address and the key. This is what lets us swap models and providers without rewriting our code.


# Let's Play Philosophy Professor

<img src="images/prof.png" width="150" alt="Prof looking at a electronic brain" style="float: left; margin-right: 15px; margin-bottom: 10px;">  In this section, to introduce ourselves to AI Agentics and see what it is, we will take on the role of a philosophy professor who must write an assignment, which they will give to their students. They will correct the answers and rank the students. In our case, all these actors will be implemented by an LLM.

We will use several LLMs, most of them free, but for a few euros you can access other services, though this is not essential for these exercises. We will use:

* Several LLMs made available to us by the University of Rennes 1
* Google's LLM
* A model running on your own machine
* and if you wish, OpenAI and Anthropic (paid: ~5€)

## Working Environment
### uv
[uv](https://docs.astral.sh/uv/) is a modern Python package and project manager. It replaces the usual combination of `venv` + `pip` (and often `pip-tools`/`poetry`) with a single, much faster tool.

Why use `uv` rather than the standard `venv` module directly:
* **One command instead of several**: with `venv` you have to create the virtual environment, activate it, then run `pip install` for every dependency yourself; `uv sync` does all of that in one go, reading the project's `pyproject.toml`.
* **Speed**: `uv`'s dependency resolver and installer are written in Rust and are typically 10-100x faster than `pip`, which matters when a whole class installs the same large dependencies (langchain, torch-based packages, etc.) at the same time.
* **A lockfile (`uv.lock`)**: it pins the exact version of every dependency (and its own dependencies), so every student ends up with the *same* working environment instead of "it works on my machine" surprises caused by `pip` silently installing newer versions.
* **Manages Python itself**: `uv` can download and manage the required Python version (here 3.12+) for you; with plain `venv` you must already have a compatible Python interpreter installed on your machine.

In short, `venv` only creates an empty virtual environment and leaves dependency management to you; `uv sync` reproduces the exact same environment for everyone from the project files, in one command.

uv is very simple and powerful. To install it:
* type `curl -LsSf https://astral.sh/uv/install.sh | sh` (you can also go to this site for more details https://docs.astral.sh/uv/getting-started/installation/)
* `uv self update` to verify that everything works
* `uv sync` to load the necessary Python modules

`uv sync` creates a `.venv` virtual environment, inside this project's folder, containing all the modules needed for this course. This venv's location differs between macOS/Linux and Windows, so instead of hunting for it in the kernel picker, register it once as a **named** Jupyter kernel — this works the same way on macOS, Linux and Windows:

* `uv run python -m ipykernel install --user --name=plidoagent --display-name="PLIDOagent"`

Then, to run the notebook cells, select this kernel in Visual Studio Code:

* click on the kernel name at the top right of the notebook, then on **Select Another Kernel...** > **Jupyter Kernel...**
* pick **PLIDOagent** from the list — this notebook is already configured to look for a kernel with this name, so it should be pre-selected or easy to spot, regardless of your OS
* if you don't see it, run the command above again and reload the kernel list (refresh icon)

If you are not using Visual Studio Code, you can instead open the notebook in a browser with `uv run jupyter lab`.

# Implementation

## API

<img src="images/api.png" width="150" alt="Prof looking at a electronic brain" style="float: left; margin-right: 15px; margin-bottom: 10px;">  To communicate with an LLM, we will use a REST API, and to be able to identify ourselves, we must create a token on its site. Our reference LLM is hosted at the University of Rennes. You can connect to it with your IMT Atlantique login.

### University of Rennes LLMs

To connect to the University of Rennes machines, click on this link: https://ragarenn.eskemm-numerique.fr/sso/ch@t/app/auth

* Select IMT Atlantique
* Log in

<img src="images/UR-param.png" width="100" alt="Prof looking at a electronic brain" style="float: left; margin-right: 15px; margin-bottom: 10px;">  At the top right, you have access to your environment. Click on **Settings**; on **Account** and on **API Keys**.

On this page you will see **two different things**, do not confuse them:
* the **JWT token**: created automatically the moment you log in, it is what keeps *your browser* authenticated while you use the RAGARENN web chat. It is tied to that login session and stops working once the session ends (e.g. when you log out or after a while) — so it is not something we create or manage ourselves, and it is useless in a script, since it would stop working the next time you run it.
* the **API key**: a separate, stable credential that *you* explicitly generate on this page, meant precisely for external programs (like our notebook) to call the server. It does not depend on a browser session and keeps working until you revoke it. This is the one we need.

* Create your **API key** and copy it.

* We now need to store this key somewhere our code can read it, without ever writing it directly in the notebook (so it doesn't end up shared or committed by accident). Create a new file named exactly `.env` at the root of this project — in Visual Studio Code: right-click the file explorer > **New File...** > type `.env`; in Jupyter: **File > New > Text File**, then rename it to `.env`. Add the line:

```
RENNES_API_KEY=<<copied key>>
```

* This project's [`.gitignore`](.gitignore) already lists `.env`, which means Git will always ignore this file: it will **never** be committed or pushed to GitHub, so your key stays on your machine and can't leak through the shared repository.



### Quick test

Before setting up Gemini and Ollama, let's immediately check that this Rennes API key actually works.

In [ ]:
# To execute this cell, choose the PLIDOagent kernel in Visual Studio

from dotenv import load_dotenv
import os

# read .env file to setup variables. override=True allows erasing old settings
load_dotenv(override=True)
rennes_api_key = os.getenv('RENNES_API_KEY')

if not rennes_api_key:
    print("Error, could not find the API key for University of Rennes")

We are going to query the University of Rennes server to find out the available models. We are going to use the OpenAI API.

In [ ]:
from openai import OpenAI

RAGARENN_BASE_URL = "https://ragarenn.eskemm-numerique.fr/sso/instance@imt/api/"

ragarenn = OpenAI(base_url=RAGARENN_BASE_URL, api_key=rennes_api_key)

try:
    models = ragarenn.models.list()
    print("Available models from ragarenn:")
    for model in models.data:
        print(model.id)
except Exception as e:
    print(f"Error fetching models: {str(e)}")

If everything is correctly configured, you should see a whole list of available models printed above. That's all we need for now: it proves your API key works and the connection to the Rennes server is properly set up. We will look later at what these sometimes strange-looking model names actually mean.

### Google Gemini

Google allows free access to its LLM, we must also retrieve an API key.

* Go to the site https://aistudio.google.com/apikey (log in to your Google account)
* Create an API key and copy it.
* Add a line in the .env file **GOOGLE_API_KEY=**<<copied key>>

### Ollama

Ollama allows you to run an LLM locally on your machine. Of course, it will be less powerful than on a specialized server, but it may be sufficient in some cases, or it will allow you to compare different models.

* Download the executable corresponding to your system from https://ollama.com/
* Install the program

Ollama works by command line, type from a shell:

* `ollama pull mistral` to install a first model
* `ollama serve` to start the server
* `ollama run mistral`
* Open a tab in your browser to verify that ollama is active http://localhost:11434. The message 'Ollama is running' should appear.

### Well-known (often paid) LLMs

If you also want to try well-known commercial models such as OpenAI's ChatGPT or Anthropic's Claude, see [paid_llms.md](paid_llms.md) for how to get an API key for each — this is optional and involves a small cost (~5€).

# First Query

<img src="images/work.png" width="150" alt="Workers" style="float: left; margin-right: 15px; margin-bottom: 10px;"> 

We are now going to query the University of Rennes server to get an answer to a question.

Even if it seems strange, we will use the OpenAI interface which has become the standard for communicating with LLM servers. Of course, we must change the default parameters to specify the URI of the University of Rennes server.

We create the message that we will send to the server, it must contain two fields:
* The `role` that we take to dialogue with it. Here, we will be a simple user `user` who questions the model.
* the content (`content`) which corresponds to the question asked.

Several parameters can be passed, they are grouped in an array.

In [ ]:
# Create the OpenAI client for the University of Rennes server
rennes = OpenAI(base_url=RAGARENN_BASE_URL, api_key=rennes_api_key)

# Question
message = [{'role':'user', 'content':"Give me a difficult question in philosophy related to artificial intelligence, "
            "without giving any answer or explanation."}]

All that remains is to query the server using the `chat.completions.create` function which will take two arguments:

* the ```model``` which is the "id" of an available model,
* the ```message``` which is the question we just formulated.

In [ ]:
response = rennes.chat.completions.create(
                model="ilaas/mistral-small-4-119b", 
                messages=message)

subject = response.choices[0].message.content

print(f"You must answer the following question: \n{subject}")

⚠️ **If this cell raises an error**, the model name is probably no longer available on the Rennes server (the list of models can change over time). Check that `"ilaas/mistral-small-4-119b"` still appears in the list of models we printed earlier in the "Quick test" section; if it doesn't, just replace it with one of the model names from that list.

Ouch ouch ouch, we have a complex question, it's time to ask other AIs to work on this assignment. We're creating a data structure to query an LLM.

In [ ]:
subject_query = [{"role": "user", "content": subject + " Give a detailed answer."}]

As we can see, there are several roles:
* `instruction` for general instructions that will indicate how the agent should process the data,
* `user` the question that is asked to the agent,
* `assistant` the agent's response.

# Querying Ollama

<img src="images/etudiants.png" width="150" alt="Students celebrating high mark" style="float: left; margin-right: 15px; margin-bottom: 10px;"> We are going to ask our local AI to think about this complex question.

We will use the same OpenAI API. You can find out the name of the models by typing in a terminal ```ollama list```

We will also use the ```Markdown``` and ```display``` functions to make the result more readable.

In [ ]:
from IPython.display import Markdown, display 

ollama=OpenAI(base_url="http://localhost:11434/v1", api_key='ollama')
model = "mistral:latest"

response = ollama.chat.completions.create(model=model, messages=subject_query)
ollama_answer = response.choices[0].message.content

display (Markdown(ollama_answer))


# Querying Gemini

No more secrets to query another server. We are going to see Gemini's opinion on the subject.

The available models are accessible here: https://ai.google.dev/gemini-api/docs?hl=fr

on the site, click on "Get an API key"

<img src=images/google_api.png>

In [ ]:
google_api_key = os.getenv('GOOGLE_API_KEY')

if not google_api_key:
    print("Error, could not find the API key from, check https://aistudio.google.com/apikey")
else:
    GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
    model = "gemini-2.5-flash-preview-05-20"
    gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
    
    response = gemini.chat.completions.create(model=model, messages=subject_query)
    gemini_answer = response.choices[0].message.content

    display (Markdown(gemini_answer))
    

Your turn to play, you can query the OpenAI and Anthropic students, provided you pay 5€. You can also change the models on RAGARENN or Gemini to compare the answers.

# Grading

<img src="images/marathon.png" width="150" alt="Winner of a marathon" style="float: left; margin-right: 15px; margin-bottom: 10px;"> Now that the papers have been submitted by our different AIs, it's time to grade them. We're going to ask the University of Rennes LLM to do the grading. It doesn't change anything from a functional point of view, you just need to formulate the question properly.

You can adapt the prompt if you have more answers.

In [ ]:
response_number=2

question_correction = f"""I am a philosophy professor, and I want to grade and 
rank the answers of these {response_number} students to the question {subject}.

The first student answered: {ollama_answer}.
The second student answered: {gemini_answer}.
"""

response = rennes.chat.completions.create(
                model="mistralai/Mistral-Small-3.1-24B-Instruct-2503", 
                messages=[{"role": "user", "content": question_correction}])

notation = response.choices[0].message.content

display(Markdown(notation))


# AI Agentics

<img src="images/victory.png" width="150" alt="Victory" style="float: left; margin-right: 15px; margin-bottom: 10px;"> Congratulations, we have taken our first steps in AI Agentics. We can describe **AI Agents** as programs that are capable of using LLMs. But not only that, as we will see later, they will be able to interact with their environment, call programs based on LLM responses, monitor physical phenomena over time, and react when the environment changes.

What we used here is a flow, quite simple and linear: we start from a question, we call several LLMs and we combine the results.

<img src="images/flow1.png" width="500">

# Interactions

<img src="images/windmill.png" width="150" alt="Wind Mill" style="float: left; margin-right: 15px; margin-bottom: 10px;">It is possible to define interactions between LLMs, for example, we can ask Gemini if the professor's answer is understandable by a 12-year-old child, and loop until it is.

We will use Gemini to judge if the answer can be understood by a 12-year-old child. To be able to process it, we will ask it to respond using a JSON structure containing an ``understand`` flag and a comment to help progress.

Notice that in the prompt we insist on having pure JSON, but LLMs are sometimes distracted and will add a Markdown marker to clearly indicate that it is JSON. Hence the loop, to only terminate the request when we have obtained the correct syntax.


In [ ]:
import json

evaluation_answer = f"""I have received this evaluation:

{notation}. 

Is it understandable by a 12-year-old child?
Respond only with a JSON Object structure, without Markdown markers, which contains an 'understand' key that indicates with 'True' that the
answer can really be understood by a 12-year-old child, and in the 'content' key give brief indications to 
improve the answer.
"""
is_json = False

while not is_json:
    understand = gemini.chat.completions.create(model=model, messages=[{"role": "user", "content": evaluation_answer}]) 

    understand_answer = understand.choices[0].message.content #text
    print (understand_answer)
    try: 
        understand_answer = json.loads(understand_answer)
        is_json=True
    except (ValueError, TypeError):
        print ("Not JSON, ask again")
        is_json=False

print (understand_answer)


Your turn to play. Ask again for a correction of the assignment with the explanations given by the **evaluator** to converge towards an answer understandable by a 12-year-old child.

<img src="images/agent_final.png" width="500">